# Clip two point clouds to their overlap and estimate initial offset (Open3D)

This notebook shows how to:
1. Load two point clouds.
2. Clip both to their shared 3D overlap extent.
3. Estimate an initial rigid offset (`4x4` transform) using FPFH + RANSAC.
4. Optionally refine with ICP.

In [ ]:
# If needed, uncomment:
# !pip install open3d numpy

In [ ]:
import copy
import numpy as np
import open3d as o3d

In [ ]:
# ---- User inputs ----
source_path = "/path/to/source.ply"
target_path = "/path/to/target.ply"

voxel_size = 0.5          # tune to your point spacing
normal_radius_factor = 2  # normal radius = factor * voxel
fpfh_radius_factor = 5    # FPFH radius  = factor * voxel

In [ ]:
def load_cloud(path: str) -> o3d.geometry.PointCloud:
    pcd = o3d.io.read_point_cloud(path)
    if pcd.is_empty():
        raise ValueError(f"Point cloud is empty or unreadable: {path}")
    return pcd


def get_overlap_aabb(a: o3d.geometry.PointCloud, b: o3d.geometry.PointCloud):
    aabb_a = a.get_axis_aligned_bounding_box()
    aabb_b = b.get_axis_aligned_bounding_box()

    min_bound = np.maximum(aabb_a.get_min_bound(), aabb_b.get_min_bound())
    max_bound = np.minimum(aabb_a.get_max_bound(), aabb_b.get_max_bound())

    if np.any(min_bound >= max_bound):
        return None, None
    return min_bound, max_bound


def crop_to_overlap(a: o3d.geometry.PointCloud, b: o3d.geometry.PointCloud):
    min_bound, max_bound = get_overlap_aabb(a, b)
    if min_bound is None:
        raise ValueError("No axis-aligned overlap between source and target bounding boxes.")

    overlap_box = o3d.geometry.AxisAlignedBoundingBox(min_bound=min_bound, max_bound=max_bound)
    a_clip = a.crop(overlap_box)
    b_clip = b.crop(overlap_box)

    if a_clip.is_empty() or b_clip.is_empty():
        raise ValueError("Overlap bbox exists, but one clipped cloud is empty. Check alignment / CRS.")

    return a_clip, b_clip, overlap_box


def preprocess_for_features(pcd: o3d.geometry.PointCloud, voxel: float):
    pcd_down = pcd.voxel_down_sample(voxel)

    normal_radius = normal_radius_factor * voxel
    fpfh_radius = fpfh_radius_factor * voxel

    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=normal_radius, max_nn=30)
    )
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=fpfh_radius, max_nn=100),
    )
    return pcd_down, fpfh

In [ ]:
# 1) Load point clouds
source = load_cloud(source_path)
target = load_cloud(target_path)

print(f"Source points: {len(source.points):,}")
print(f"Target points: {len(target.points):,}")

In [ ]:
# 2) Clip both clouds to overlap extent
source_clip, target_clip, overlap_box = crop_to_overlap(source, target)

print(f"Clipped source points: {len(source_clip.points):,}")
print(f"Clipped target points: {len(target_clip.points):,}")
print("Overlap min bound:", overlap_box.get_min_bound())
print("Overlap max bound:", overlap_box.get_max_bound())

In [ ]:
# Optional: visualize clipped overlap
source_vis = copy.deepcopy(source_clip)
target_vis = copy.deepcopy(target_clip)
source_vis.paint_uniform_color([1, 0, 0])
target_vis.paint_uniform_color([0, 1, 0])

o3d.visualization.draw_geometries([source_vis, target_vis], window_name="Clipped overlap (red=source, green=target)")

In [ ]:
# 3) Estimate initial offset (global registration)
source_down, source_fpfh = preprocess_for_features(source_clip, voxel_size)
target_down, target_fpfh = preprocess_for_features(target_clip, voxel_size)

distance_threshold = 1.5 * voxel_size

result_ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_down,
    target_down,
    source_fpfh,
    target_fpfh,
    mutual_filter=True,
    max_correspondence_distance=distance_threshold,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    ransac_n=4,
    checkers=[
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold),
    ],
    criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(100000, 0.999),
)

print("Initial transformation (source -> target):")
print(result_ransac.transformation)
print("RANSAC fitness:", result_ransac.fitness)
print("RANSAC inlier RMSE:", result_ransac.inlier_rmse)

In [ ]:
# 4) Optional refinement with ICP
icp_threshold = 1.0 * voxel_size
result_icp = o3d.pipelines.registration.registration_icp(
    source_clip,
    target_clip,
    max_correspondence_distance=icp_threshold,
    init=result_ransac.transformation,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
)

print("Refined transformation (ICP):")
print(result_icp.transformation)
print("ICP fitness:", result_icp.fitness)
print("ICP inlier RMSE:", result_icp.inlier_rmse)

In [ ]:
# Visualize final alignment
aligned_source = copy.deepcopy(source_clip)
aligned_source.transform(result_icp.transformation)
aligned_source.paint_uniform_color([0, 0, 1])
target_vis2 = copy.deepcopy(target_clip)
target_vis2.paint_uniform_color([0, 1, 0])

o3d.visualization.draw_geometries([aligned_source, target_vis2], window_name="Aligned overlap (blue=source transformed, green=target)")

## Notes
- If registration fails, try larger `voxel_size` and thresholds.
- If clouds are already close, use smaller values for finer alignment.
- This uses AABB intersection for overlap. If clouds are strongly rotated, a coarse pre-alignment may help first.